<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

در این مثال:
1. سؤال کاربر به LLM داده می‌شود.
2. LLM سه Query جایگزین تولید می‌کند.
3. برای هر Query، Semantic Search اجرا می‌شود.
4. نتایج همه جستجوها ترکیب می‌شوند.
5. اسناد تکراری حذف می‌شوند.
از همان OpenAI-compatible API در LM Studio استفاده می‌کنیم.

In [1]:
import re
import json
import numpy as np

from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. LM Studio Client
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

MODEL_ID = "google/gemma-4-e4b"

# 2. HR Knowledge Base
documents = [
    {"id": "HR-001", "text": "Employees receive 26 annual leave days per year."},
    {"id": "HR-002", "text": "Leave requests must be approved by the manager."},
    {"id": "HR-003", "text": "Employees have supplementary health insurance."},
    {"id": "HR-004", "text": "Monthly salaries are paid at the end of each month."},
    {"id": "HR-005", "text": "Employees can work remotely two days per week."},
    {"id": "HR-006", "text": "Employees can request unpaid leave."},
]

# 3. Initialize Embedding Model
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

document_embeddings = embedding_model.encode(
    [doc["text"] for doc in documents], convert_to_numpy=True, normalize_embeddings=True
)


# 4. Generate Multiple Queries
def generate_queries(original_query, num_queries=3):

    response = client.chat.completions.create(
        model=MODEL_ID,
        temperature=0.3,
        messages=[
            {
                "role": "system",
                "content": f"""
                You are a query generation component for an HR RAG system.

                Generate exactly {num_queries} alternative search queries
                based on the user's original question.

                Rules:
                - Preserve the original intent.
                - Use different wording and relevant terminology.
                - Do not answer the question.
                - Do not invent company policies.
                - Return only a valid JSON array of strings.
                - Write all queries in English.
                """,
            },
            {"role": "user", "content": original_query},
        ],
    )

    content = response.choices[0].message.content.strip()

    # Extract JSON array if the model adds surrounding text
    match = re.search(r"\[[\s\S]*\]", content)

    if not match:
        raise ValueError("The model did not return a valid JSON array.")

    queries = json.loads(match.group())

    if not isinstance(queries, list) or not all(isinstance(q, str) for q in queries):
        raise ValueError("Invalid query list.")

    return queries


# 5. Semantic Search
def semantic_search(query, top_k=3):

    query_embedding = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )

    scores = cosine_similarity(query_embedding, document_embeddings)[0]

    indices = np.argsort(scores)[::-1][:top_k]

    return [{**documents[index], "score": float(scores[index])} for index in indices]


# 6. Multi-Query Retrieval
def multi_query_retrieval(original_query, top_k=3):

    generated_queries = generate_queries(original_query)

    # Include original query in retrieval
    all_queries = [original_query] + generated_queries

    retrieved_documents = {}
    query_results = {}

    for query in all_queries:
        results = semantic_search(query, top_k=top_k)

        query_results[query] = results

        for result in results:
            doc_id = result["id"]

            if doc_id not in retrieved_documents:
                retrieved_documents[doc_id] = {**result, "matched_queries": [query]}
            else:
                retrieved_documents[doc_id]["matched_queries"].append(query)

    return generated_queries, query_results, list(retrieved_documents.values())


# 7. Execute Pipeline
user_query = "How can employees take time off from work?"

generated_queries, query_results, final_results = multi_query_retrieval(
    user_query, top_k=3
)

print("Original Query:")
print(user_query)

print("\nGenerated Queries:")
for query in generated_queries:
    print("-", query)

print("\nIndividual Search Results:")

for query, results in query_results.items():
    print(f"\nQuery: {query}")

    for result in results:
        print(result["id"], round(result["score"], 4))

print("\nMerged Results:")

for result in final_results:
    print("\nID:", result["id"])
    print("Text:", result["text"])
    print("Matched Queries:", len(result["matched_queries"]))
    print("Queries:", result["matched_queries"])

e:\AI-Engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2890.35it/s]


Original Query:
How can employees take time off from work?

Generated Queries:
- Employee guide for requesting time off
- What is the procedure for taking paid leave?
- How to apply for vacation or personal days?

Individual Search Results:

Query: How can employees take time off from work?
HR-006 0.5832
HR-001 0.5397
HR-005 0.5074

Query: Employee guide for requesting time off
HR-006 0.6274
HR-002 0.5311
HR-005 0.5157

Query: What is the procedure for taking paid leave?
HR-006 0.7696
HR-001 0.556
HR-002 0.5157

Query: How to apply for vacation or personal days?
HR-005 0.3945
HR-006 0.3052
HR-001 0.2738

Merged Results:

ID: HR-006
Text: Employees can request unpaid leave.
Matched Queries: 4
Queries: ['How can employees take time off from work?', 'Employee guide for requesting time off', 'What is the procedure for taking paid leave?', 'How to apply for vacation or personal days?']

ID: HR-001
Text: Employees receive 26 annual leave days per year.
Matched Queries: 3
Queries: ['How can e